In [9]:
import pandas as pd
import sys
import os
import datetime
import time
import numpy as np
import json
import pprint
import _pickle as pickle

sys.path.append("/home/kp_info_loader")


In [2]:
with open("/home/kp_info_loader/kp_info_loader_config.json") as f:
    config = json.load(f)

In [16]:
enabled_market_klines = config['node_settings']['kline_generator1']['enabled_market_klines']
enabled_market_klines

['BITHUMB_SPOT/KRW:BINANCE_SPOT/USDT',
 'BITHUMB_SPOT/BTC:BINANCE_USD_M/USDT',
 'BYBIT_USD_M/USDT:BINANCE_USD_M/USDT',
 'BYBIT_USD_M/USDT:OKX_USD_M/USDT']

In [19]:
def generate_enabled_websocket_list(enabled_market_klines):
    market_list = []
    for each_market_combi in enabled_market_klines:
        market_code1, market_code2 = each_market_combi.split(':')
        market_list.append(market_code1.split('/')[0])
        market_list.append(market_code2.split('/')[0])
    market_list = list(set(market_list))
    return market_list
generate_enabled_websocket_list(enabled_market_klines)

['OKX_USD_M', 'BITHUMB_SPOT', 'BINANCE_SPOT', 'BINANCE_USD_M', 'BYBIT_USD_M']

In [15]:
generate_enabled_websocket_list(enabled_market_klines)



def generate_enabled_market_code_dict(enabled_market_klines):
    organized_markets = {}

    def add_market_product(market, product_type, product):
        if market not in organized_markets:
            organized_markets[market] = {"SPOT": [], "USD_M": {"PERPETUAL": [], "FUTURES": []}, "COIN_M": {"PERPETUAL": [], "FUTURES": []}}
        
        if product_type == "SPOT":
            if product not in organized_markets[market]["SPOT"]:
                organized_markets[market]["SPOT"].append(product)
        elif product_type in ["USD_M", "COIN_M"]:
            if product not in organized_markets[market][product_type]["PERPETUAL"]:
                organized_markets[market][product_type]["PERPETUAL"].append(product)

    # Process each entry in the enabled_market_klines
    for entry in enabled_market_klines:
        pairs = entry.split(":")
        for pair in pairs:
            market, quote_asset = pair.split("/")
            # Handle different market types
            if "USD_M" in market:
                market_name = market.replace("_USD_M", "")
                market_type = "USD_M"
            elif "COIN_M" in market:
                market_name = market.replace("_COIN_M", "")
                market_type = "COIN_M"
            else:
                market_name, market_type = market.split("_")
            add_market_product(market_name, market_type, quote_asset)
    return organized_markets

pprint.pprint(generate_enabled_market_code_dict(enabled_market_klines))

{'BINANCE': {'COIN_M': {'FUTURES': [], 'PERPETUAL': []},
             'SPOT': ['USDT'],
             'USD_M': {'FUTURES': [], 'PERPETUAL': ['USDT']}},
 'UPBIT': {'COIN_M': {'FUTURES': [], 'PERPETUAL': []},
           'SPOT': ['KRW', 'BTC'],
           'USD_M': {'FUTURES': [], 'PERPETUAL': []}}}


In [7]:
'BINANCE_USD_M'.find("_")

7

In [6]:
pairs

['BITHUMB_SPOT/BTC', 'BINANCE_USD_M/USDT']